# Kotlin Scope Functions

Scope functions let you execute a block of code **in the context of an object**.  
There are **5 scope functions**, each with subtle but important differences.

| Function | Object ref | Return value | Is extension? |
|----------|-----------|--------------|---------------|
| `let`   | `it`       | Lambda result | ✅ Yes        |
| `run`   | `this`     | Lambda result | ✅ Yes        |
| `with`  | `this`     | Lambda result | ❌ No         |
| `apply` | `this`     | Context object| ✅ Yes        |
| `also`  | `it`       | Context object| ✅ Yes        |

---
## 1️⃣ `let` — Transform or null-check an object
- Refers to object as **`it`**
- Returns **lambda result**
- Best for **null safety checks** and **chaining transformations**

In [ ]:
// Basic let usage
val name: String? = "John Doe"

val length = name?.let {
    println("Name is: $it")
    it.length  // this is returned
}

println("Length: $length")

In [ ]:
// Null-safe chain using let
val nullableName: String? = null

val result = nullableName?.let {
    "Hello, $it!"
} ?: "Name was null — using default"

println(result)

In [ ]:
// Chaining transformations
val transformed = "  hello world  "
    .let { it.trim() }
    .let { it.uppercase() }
    .let { "Result: $it" }

println(transformed)

---
## 2️⃣ `run` — Configure + compute a result
- Refers to object as **`this`**
- Returns **lambda result**
- Best when you need to **both configure AND compute a result**

In [ ]:
data class DatabaseConfig(
    var host: String = "",
    var port: Int = 0,
    var dbName: String = ""
) {
    fun connectionString() = "jdbc:postgresql://$host:$port/$dbName"
}

val connString = DatabaseConfig().run {
    host = "localhost"
    port = 5432
    dbName = "mydb"
    connectionString()  // computed result is returned
}

println(connString)

In [ ]:
// run as standalone block (non-extension) for grouping logic
val token = run {
    val header  = "eyJhbGciOiJIUzI1NiJ9"
    val payload = "eyJ1c2VySWQiOiIxMjMifQ"
    "$header.$payload"  // returned
}

println("Token: $token")

---
## 3️⃣ `with` — Operate on an object (non-extension)
- Refers to object as **`this`**
- Returns **lambda result**
- Object is passed **as an argument** (not chained)
- Best for **calling multiple methods** on a non-null object

In [ ]:
data class User(val name: String, val age: Int, val email: String)

val user = User("Alice", 30, "alice@example.com")

val summary = with(user) {
    println("Name  : $name")
    println("Age   : $age")
    println("Email : $email")
    "Summary → ${name}, ${age} yrs"   // returned
}

println(summary)

In [ ]:
// with for building formatted output
data class Product(val id: Int, val name: String, val price: Double, val stock: Int)

val product = Product(101, "Kotlin in Action", 39.99, 250)

val report = with(product) {
    """
    📦 Product Report
    -----------------
    ID    : $id
    Name  : $name
    Price : $$price
    Stock : $stock units
    Status: ${if (stock > 0) "✅ In Stock" else "❌ Out of Stock"}
    """.trimIndent()
}

println(report)

---
## 4️⃣ `apply` — Configure / Builder pattern
- Refers to object as **`this`**
- Returns **the context object itself**
- Best for **object initialization** and **builder-style** setup

In [ ]:
data class ServerConfig(
    var host: String = "",
    var port: Int = 0,
    var timeout: Int = 0,
    var retryCount: Int = 0,
    var useSsl: Boolean = false
)

val config = ServerConfig().apply {
    host       = "api.prod.example.com"
    port       = 443
    timeout    = 30_000
    retryCount = 3
    useSsl     = true
}  // apply returns the ServerConfig object itself

println(config)

In [ ]:
// apply in a chain — configure then use
data class HttpRequest(
    var url: String = "",
    var method: String = "GET",
    val headers: MutableMap<String, String> = mutableMapOf(),
    var body: String? = null
) {
    fun execute() = "Executing $method → $url with ${headers.size} header(s)"
}

val response = HttpRequest()
    .apply {
        url    = "https://api.example.com/users"
        method = "POST"
        headers["Authorization"] = "Bearer abc123"
        headers["Content-Type"]  = "application/json"
        body   = """{"name": "Alice"}"""
    }
    .execute()

println(response)

---
## 5️⃣ `also` — Side effects, logging, debugging
- Refers to object as **`it`**
- Returns **the context object itself**
- Best for **side effects** (logging, validation, debugging) **without modifying** the object

In [ ]:
// also for logging in a chain without breaking it
data class Order(val id: String, val amount: Double, var status: String = "PENDING")

fun processOrder(order: Order): Order {
    return order
        .also { println("[LOG] Processing order: ${it.id}") }
        .also { require(it.amount > 0) { "Amount must be positive" } }
        .also { it.status = "PROCESSED" }
        .also { println("[LOG] Order done: ${it.id} → ${it.status}") }
}

val order = Order("ORD-001", 149.99)
val processed = processOrder(order)
println(processed)

In [ ]:
// also for debug-printing mid-chain
val numbers = mutableListOf(3, 1, 4, 1, 5, 9, 2, 6)
    .also { println("Original : $it") }
    .filter { it > 2 }
    .also { println("Filtered : $it") }
    .sorted()
    .also { println("Sorted   : $it") }

---
## 🔗 Real-World Combined Example

Combining all 5 scope functions in a realistic API response pipeline.

In [ ]:
data class ApiResponse(val data: String?, val error: String?)
data class UserDto(val name: String, var source: String = "", var timestamp: Long = 0L)

fun parseJson(raw: String) = UserDto(name = raw.substringAfter("name:").trim())

fun handleResponse(response: ApiResponse?) {
    val result = response
        ?.also  { println("[also]  Raw response: $it") }           // side-effect
        ?.let   { it.data ?: throw Exception("Error: ${it.error}") } // null-safe transform
        ?.run   { parseJson(this) }                                  // configure + compute
        ?.apply {                                                     // configure object
            source    = "REST_API"
            timestamp = System.currentTimeMillis()
        }
        ?.also  { println("[also]  Final object ready: $it") }      // side-effect

    println("Result → $result")
}

// Test with valid response
handleResponse(ApiResponse(data = "name: Alice", error = null))

println()

// Test with null data
try {
    handleResponse(ApiResponse(data = null, error = "Not Found"))
} catch (e: Exception) {
    println("Caught: ${e.message}")
}

---
## 🧭 Decision Guide

```
Need null safety / transform?       →  let    (?. let { it })
Configuring object, no result?      →  apply  (returns object)
Configuring + need a result?        →  run    (returns lambda result)
Multiple calls, object as argument? →  with   (non-extension)
Side effects / logging in chain?    →  also   (returns object)
```

### ⚠️ Best Practices
- **`apply` & `also`** → return the **object** → great for chaining setup  
- **`let`, `run` & `with`** → return the **lambda result** → great for transformation  
- **`this` receivers** (`apply`, `run`, `with`) → cleaner when accessing many properties  
- **`it` receivers** (`let`, `also`) → cleaner when passing to other functions  
- Avoid **nesting scope functions** more than 2 levels — it tanks readability 🚫